# M5.4 · GNN: message passing

Graf üzerinde öğrenme. 3 düğümlü bir yolda normalize komşuluk matrisini ve bir GCN katmanını elle hesaplıyor, saf Python'da kenar kenar mesaj geçişi yazıyor, NumPy'da aynı işi yoğun matrisle ve scatter-add ile döngüsüz yapıyor, PyTorch'ta `index_add_` ile bir GCN kurup az etiketli bir topluluk grafında MLP'yi açık ara geçtiğini görüyor ve en sonda yüzlerce küçük grafı tek bir dev grafa birleştirip (PyG tarzı batching) graf düzeyinde sınıflandırmayı döngüsüz yapıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m5-4-gnn.html](https://m-gonul.github.io/pytorch-lab/m5-4-gnn.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m5_4_gnn.py
CHECKS = {
 "b": "import numpy as np\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef dense(n, edges):\n    A = np.eye(n)\n    for i, j in edges:\n        A[i, j] = A[j, i] = 1\n    d = A.sum(1)\n    return A / np.sqrt(d[:, None] * d[None, :])\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def w():\n        W = need(ns, \"gcn_weights\")(3, [(0, 1), (1, 2)])\n        ensure(len(W) == 7, f\"3 öz-döngü + 2 kenar × 2 yön = 7 ağırlık; {len(W)} var\")\n        close([W[(0, 0)], W[(0, 1)], W[(1, 0)], W[(1, 1)]], [0.5, 1 / 6 ** 0.5, 1 / 6 ** 0.5, 1 / 3])\n    R.test(\"gcn_weights (elle örnek)\", w)\n    R.test(\"gcn_layer (elle örnek)\", lambda: close(need(ns, \"gcn_layer\")([[1.0], [2.0], [3.0]], 3, [(0, 1), (1, 2)], [[1.0]]), [[1.316497], [2.29966], [2.316497]], atol=1e-5))\n\n    def rnd():\n        rng = np.random.default_rng(0)\n        n = 7; edges = [(0, 1), (0, 2), (1, 2), (2, 3), (4, 5)]\n        X = rng.normal(size=(n, 3)); W = rng.normal(size=(3, 2))\n        close(need(ns, \"gcn_layer\")(X.tolist(), n, edges, W.tolist()), dense(n, edges) @ X @ W, what=\"izole düğüm 6 dahil\")\n    R.test(\"rastgele graf = yoğun Â X W\", rnd)\n\n    def mn():\n        out = need(ns, \"mean_neighbors\")([[1.0, 0], [2.0, 1], [3.0, 2], [9.0, 9]], 4, [(0, 1), (1, 2), (0, 2)])\n        close(out, [[2.5, 1.5], [2, 1], [1.5, 0.5], [0, 0]])\n    R.test(\"mean_neighbors (izole düğüm → 0)\", mn)\n",
 "c": "import numpy as np\nfrom labcheck import close, no_loops, no_imports, need, ensure\n\n\ndef rand_graph(rng, n, p):\n    A = np.triu(rng.random((n, n)) < p, 1)\n    A = A | A.T\n    return np.array(np.nonzero(A))\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(0)\n    ei = rand_graph(rng, 30, 0.15); X = rng.normal(size=(30, 4))\n\n    def dn():\n        A = np.eye(30)\n        for s, t in ei.T:\n            A[s, t] = 1\n        d = A.sum(1)\n        close(need(ns, \"dense_norm_adj\")(ei, 30), A / np.sqrt(np.outer(d, d)))\n    R.test(\"dense_norm_adj\", dn)\n    R.test(\"propagate = Â X (scatter-add)\", lambda: close(need(ns, \"propagate\")(ei, X, 30), need(ns, \"dense_norm_adj\")(ei, 30) @ X))\n\n    def perm():\n        p = rng.permutation(30); inv = np.argsort(p)\n        ei2 = inv[ei]\n        X2 = X[p]\n        close(need(ns, \"propagate\")(ei2, X2, 30), ns[\"propagate\"](ei, X, 30)[p], what=\"yeniden numaralandırma\")\n    R.test(\"permütasyon eşdeğişkenliği\", perm)\n\n    def iso():\n        out = need(ns, \"propagate\")(np.zeros((2, 0), int), X[:5], 5)\n        close(out, X[:5], what=\"kenarsız graf: Â = I\")\n    R.test(\"kenarsız graf\", iso)\n",
 "d": "import torch\nimport torch.nn.functional as F\nfrom torch import nn\nfrom labcheck import close, need, ensure\n\n\ndef sbm(seed):\n    g = torch.Generator().manual_seed(seed)\n    C, n = 3, 40\n    N = C * n\n    y = torch.arange(C).repeat_interleave(n)\n    P = torch.where(y[:, None] == y[None, :], 0.15, 0.01)\n    A = (torch.rand(N, N, generator=g) < P).float().triu(1)\n    A = A + A.T\n    ei = A.nonzero().T\n    X = torch.cat([F.one_hot(y, C).float() + 1.5 * torch.randn(N, C, generator=g), torch.randn(N, 13, generator=g)], 1)\n    tr = torch.zeros(N, dtype=torch.bool)\n    for c in range(C):\n        tr[(y == c).nonzero()[:5, 0]] = True\n    return X, ei, y, tr, A\n\n\nclass MLP(nn.Module):\n    def __init__(self, i, h, o):\n        super().__init__(); self.l1 = nn.Linear(i, h); self.l2 = nn.Linear(h, o)\n\n    def forward(self, x, ei):\n        return self.l2(F.dropout(F.relu(self.l1(x)), 0.5, self.training))\n\n\ndef check(ns, R):\n    X, ei, y, tr, A = sbm(0)\n    N = X.shape[0]\n\n    def prop():\n        At = A + torch.eye(N); d = At.sum(1)\n        Ah = At / (d[:, None] * d[None, :]).sqrt()\n        x = torch.randn(N, 4, dtype=torch.float64)\n        close(need(ns, \"gcn_propagate\")(x, ei, N), Ah.double() @ x)\n    R.test(\"gcn_propagate = Â x\", prop)\n\n    def grad():\n        x = torch.randn(N, 3, requires_grad=True)\n        need(ns, \"gcn_propagate\")(x, ei, N).sum().backward()\n        ensure(x.grad is not None and x.grad.abs().sum() > 0, \"x'e gradyan akmalı\")\n    R.test(\"gradyan akıyor\", grad)\n\n    def train():\n        accs = {}\n        for name, M in ((\"GCN\", need(ns, \"GCN\", \"sınıf\")), (\"MLP\", MLP)):\n            torch.manual_seed(0)\n            m = M(16, 16, 3)\n            m = need(ns, \"train_node_clf\")(m, X, ei, y, tr) or m\n            m.eval()\n            with torch.no_grad():\n                accs[name] = (m(X, ei).argmax(1) == y)[~tr].float().mean().item()\n        ensure(accs[\"GCN\"] > 0.8, f\"GCN test doğruluğu {accs['GCN']:.3f}\")\n        ensure(accs[\"GCN\"] > accs[\"MLP\"] + 0.2, \"GCN, MLP'yi açıkça geçmeli\")\n        return f\"15 etiketle test doğruluğu: GCN {accs['GCN']:.3f}, MLP {accs['MLP']:.3f}\"\n    R.test(\"az etiketle düğüm sınıflandırma\", train)\n",
 "e": "import torch\nfrom torch import nn\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef rand_graphs(g, k, d):\n    out = []\n    for _ in range(k):\n        n = int(torch.randint(3, 25, (1,), generator=g))\n        A = (torch.rand(n, n, generator=g) < 0.25).triu(1)\n        A = A | A.T\n        out.append((torch.randn(n, d, generator=g, dtype=torch.float64), A.nonzero().T))\n    return out\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"collate: en fazla tek seviye döngü (graflar)\", lambda: max_loops(src, 1, func=\"collate\"))\n    R.test(\"readout döngüsüz\", lambda: max_loops(src, 0, func=\"readout\"))\n    g = torch.Generator().manual_seed(0)\n    graphs = rand_graphs(g, 20, 4)\n    prop = need(ns, \"propagate\")\n\n    def col():\n        X, EI, batch = need(ns, \"collate\")(graphs)\n        ensure(X.shape[0] == sum(x.shape[0] for x, _ in graphs) and EI.shape[1] == sum(e.shape[1] for _, e in graphs), \"toplam düğüm/kenar sayısı\")\n        ensure(torch.equal(batch[EI[0]], batch[EI[1]]), \"hiçbir kenar iki grafı birbirine bağlamamalı\")\n        ensure(torch.equal(torch.bincount(batch), torch.tensor([x.shape[0] for x, _ in graphs])), \"batch vektörü\")\n    R.test(\"collate: blok-köşegen graf\", col)\n\n    def ro():\n        x = torch.tensor([[1.0, -1], [2, -2], [5, 0], [0, 3], [1, 1]], dtype=torch.float64)\n        close(need(ns, \"readout\")(x, torch.tensor([0, 0, 1, 1, 1]), 2), [[1.5, -1.5, 2, -1], [2, 4 / 3, 5, 3]])\n    R.test(\"readout: [ortalama, max] (negatif değerler dahil)\", ro)\n\n    W1 = torch.randn(4, 16, generator=g, dtype=torch.float64); W2 = torch.randn(16, 16, generator=g, dtype=torch.float64)\n\n    def emb_one(x, ei):\n        h = torch.relu(prop(x @ W1, ei, x.shape[0]))\n        h = prop(h @ W2, ei, x.shape[0])\n        return torch.cat([h.mean(0), h.max(0).values])\n\n    def eq():\n        X, EI, batch = need(ns, \"collate\")(graphs)\n        h = torch.relu(prop(X @ W1, EI, X.shape[0]))\n        h = prop(h @ W2, EI, X.shape[0])\n        close(need(ns, \"readout\")(h, batch, len(graphs)), torch.stack([emb_one(x, e) for x, e in graphs]))\n    R.test(\"birleşik graf = tek tek işlenen graflar\", eq)\n    if \"readout\" not in ns or \"collate\" not in ns:\n        return\n    big = rand_graphs(g, 500, 4)\n    big = [(x.float(), e) for x, e in big]\n    W1f, W2f = W1.float(), W2.float()\n\n    def loop_all():\n        outs = []\n        for x, e in big:\n            h = torch.relu(prop(x @ W1f, e, x.shape[0])); h = prop(h @ W2f, e, x.shape[0])\n            outs.append(torch.cat([h.mean(0), h.max(0).values]))\n        return torch.stack(outs)\n\n    def batched():\n        X, EI, b = ns[\"collate\"](big)\n        h = torch.relu(prop(X @ W1f, EI, X.shape[0])); h = prop(h @ W2f, EI, X.shape[0])\n        return ns[\"readout\"](h, b, len(big))\n    t_loop = bench(loop_all, repeat=3, number=1)\n    t_b = bench(batched, repeat=3)\n    R.table(\"500 graf, 2 katman GCN + readout\", [\"yöntem\", \"süre\"], [[\"graf başına döngü\", fmt_time(t_loop)], [\"collate + tek geçiş\", fmt_time(t_b)]])\n"
}

In [ ]:
import importlib, labcheck, checks_m5_4_gnn as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Bir grafın düğüm özellikleri $X\in\mathbb R^{N\times d}$, kenarları `edge_index` $\in\mathbb N^{2\times E}$ (satır 0 kaynak, satır 1 hedef; yönsüz graf için iki yön de listelenir).

**message passing:** her düğüm komşularından mesaj toplar ve kendini günceller:

$$
h_i'=\phi\Big(h_i,\ \bigoplus_{j\in\mathcal N(i)}\psi(h_i,h_j)\Big),\qquad \bigoplus\in\{\textstyle\sum,\text{ort},\max\}.
$$

**GCN** (Kipf & Welling) bunun doğrusal, normalize edilmiş hâli:

$$
H'=\sigma\big(\hat A\,H\,W\big),\qquad \hat A=\tilde D^{-1/2}(A+I)\tilde D^{-1/2},\quad \tilde D_{ii}=1+\deg(i).
$$

Kenar başına bakınca: $\ h_i'=\sum_{j\in\mathcal N(i)\cup\{i\}}\frac{1}{\sqrt{\tilde d_i\tilde d_j}}\,(Wh_j)$. Bu toplam, kenar listesi üzerinde bir **scatter-add**'dir; yoğun $N\times N$ matris hiç kurulmaz.

## 1 · Elle

Öz-döngü olmadan düğüm kendi özelliğini unutur; normalizasyon olmadan yüksek dereceli düğümlerin değerleri her katmanda büyür. $1/\sqrt{\tilde d_i\tilde d_j}$ ikisini dengeleyen simetrik seçim; $\hat A$'nın özdeğerleri $(-1,1]$ aralığında kalır.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Yol grafı 1 – 2 – 3, öz-döngülerle: $\tilde A=A+I$, dereceler $\tilde d=(2,3,2)$, $\hat A=\tilde D^{-1/2}\tilde A\tilde D^{-1/2}$**

- `a11`: $\hat A_{11}$
- `a12`: $\hat A_{12}$
- `a22`: $\hat A_{22}$
- `a13`: $\hat A_{13}$

**Özellikler $X=(1,2,3)^\top$, $W=1$: $H=\hat AXW$**

- `h1`: $H_1$
- `h2`: $H_2$
- `mean2`: Yalnız komşu ortalaması (öz-döngüsüz): düğüm 2

In [ ]:
cevaplar = dict(
    a11=None,
    a12=None,
    a22=None,
    a13=None,
    h1=None,
    h2=None,
    mean2=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"a11":[0.5,0],"a12":[0.408248,0.0005],"a22":[0.333333,0.0005],"a13":[0,0],"h1":[1.316497,0.0005],"h2":[2.29966,0.0005],"mean2":[2,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python; `edges` yönsüz kenar listesi `[(0, 1), (1, 2)]` (her kenar bir kez), düğümler `0..n-1`, `X` liste listesi, `W` liste listesi (`d_in × d_out`).

- `gcn_weights(n, edges)` → `{(i, j): ağırlık}` sözlüğü; öz-döngüler ve **iki yön** dahil.
- `gcn_layer(X, n, edges, W)` → $\hat AXW$: önce her düğüm için $XW$, sonra her `(i, j)` için `out[i] += w_ij * XW[j]`.
- `mean_neighbors(X, n, edges)` — öz-döngüsüz komşu ortalaması (komşusu yoksa sıfır vektör).

In [ ]:
%%writefile stage_b.py
import math


def gcn_weights(n, edges):
    ...


def gcn_layer(X, n, edges, W):
    ...


def mean_neighbors(X, n, edges):
    ...


print(gcn_layer([[1.0], [2.0], [3.0]], 3, [(0, 1), (1, 2)], [[1.0]]))   # [[1.3165], [2.2997], [2.3165]]
print(mean_neighbors([[1.0], [2.0], [3.0]], 3, [(0, 1), (1, 2)]))      # [[2], [2], [2]]

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy; `edge_index: (2, E)` iki yönlü, öz-döngüsüz. **Döngü yok.**

- `dense_norm_adj(edge_index, n)` → $\hat A$ (yoğun): `A[src, dst] = 1`, `+ I`, derece, normalize.
- `propagate(edge_index, X, n)` → $\hat AX$ yoğun matris kurmadan: öz-döngüleri ekle, dereceyi `np.bincount` ile al, kenar ağırlıkları $1/\sqrt{\tilde d_s\tilde d_t}$, `np.add.at(out, dst, w[:, None] * X[src])`.

Kontrol ikisinin aynı olduğunu ve permütasyon eşdeğişkenliğini sınar.

In [ ]:
%%writefile stage_c.py
import numpy as np


def dense_norm_adj(edge_index, n):
    ...


def propagate(edge_index, X, n):
    ...


ei = np.array([[0, 1, 1, 2], [1, 0, 2, 1]])
X = np.array([[1.0], [2.0], [3.0]])
print(dense_norm_adj(ei, 3) @ X, propagate(ei, X, 3))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

- `gcn_propagate(x, edge_index, N)` — c'deki `propagate`'in torch hâli (`index_add_`); gradyan akmalı.
- `GCN(d_in, hidden, d_out, dropout=0.5)` — `lin1`, `lin2` (`nn.Linear`); `forward(x, edge_index)`: $\hat A\,\text{lin1}(x)$ → ReLU → dropout → $\hat A\,\text{lin2}(h)$ (logit).
- `train_node_clf(model, X, ei, y, train_mask, steps=200, lr=0.01)` — Adam (`weight_decay=5e-4`), kayıp yalnız `train_mask` düğümlerinde.

Kontrol 3 topluluklu bir stokastik blok modelinde sınıf başına **5 etiketle** eğitir ve aynı özelliklerle bir MLP'ye karşı karşılaştırır.

In [ ]:
%%writefile stage_d.py
import torch
import torch.nn.functional as F
from torch import nn


def gcn_propagate(x, edge_index, N):
    ...


class GCN(nn.Module):
    def __init__(self, d_in, hidden, d_out, dropout=0.5):
        super().__init__()
        ...

    def forward(self, x, edge_index):
        ...


def train_node_clf(model, X, ei, y, train_mask, steps=200, lr=0.01):
    ...


ei = torch.tensor([[0, 1, 1, 2], [1, 0, 2, 1]])
print(gcn_propagate(torch.tensor([[1.0], [2.0], [3.0]]), ei, 3))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Graf sınıflandırma için PyG tarzı batching. `propagate` hazır verildi.

- `collate(graphs)` — `graphs`: `(x_i, edge_index_i)` listesi → `(X, EI, batch)`. Düğümleri art arda ekle, her grafın kenarlarına o grafın **düğüm ofsetini** ekle (`cumsum`), `batch[k]` = düğüm $k$'nin grafı (`repeat_interleave`). Graflar üzerinde tek seviye döngü/comprehension serbest.
- `readout(x, batch, B)` → `(B, 2d)`: graf başına `[ortalama, max]`. **Döngü yok:** `index_add_` ve `scatter_reduce` (ya da `index_reduce_`, `"amax"`).

Kontrol, birleşik grafla hesaplanan graf temsillerinin grafları tek tek işlemekle birebir aynı olduğunu doğrular.

In [ ]:
%%writefile stage_e.py
import torch


def propagate(x, edge_index, N):
    loops = torch.arange(N)
    src, dst = torch.cat([edge_index[0], loops]), torch.cat([edge_index[1], loops])
    deg = torch.zeros(N, dtype=x.dtype).index_add_(0, dst, torch.ones(dst.shape[0], dtype=x.dtype))
    return torch.zeros_like(x).index_add_(0, dst, x[src] * (deg[src] * deg[dst]).rsqrt()[:, None])


# ---- senin kodun ----

def collate(graphs):
    """[(x_i, ei_i)] → (X, EI, batch). Graflar üzerinde tek seviye döngü serbest."""
    ...


def readout(x, batch, B):
    """→ (B, 2d): [ortalama, max]. Döngü yok."""
    ...


g1 = (torch.tensor([[1.0], [2.0]]), torch.tensor([[0, 1], [1, 0]]))
g2 = (torch.tensor([[5.0], [0.0], [1.0]]), torch.tensor([[0, 1], [1, 0]]))
X, EI, batch = collate([g1, g2])
print(EI, batch)                       # kenarlar [[0,1,2,3],[1,0,3,2]], batch [0,0,1,1,1]
print(readout(X, batch, 2))            # [[1.5, 2], [2, 5]]

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Kaynaklar

- [Kipf & Welling — Semi-Supervised Classification with Graph Convolutional Networks](https://arxiv.org/abs/1609.02907) — arXiv
- [Gilmer ve ark. — Neural Message Passing for Quantum Chemistry](https://arxiv.org/abs/1704.01212) — arXiv
- [UvA Deep Learning Tutorials — Graph Neural Networks](https://uvadlc-notebooks.readthedocs.io/) — MIT
- [Stanford CS224W — Machine Learning with Graphs](https://web.stanford.edu/class/cs224w/) — doğrulanmadı
- [PyTorch Geometric — Creating Message Passing Networks](https://pytorch-geometric.readthedocs.io/) — MIT